# **Data Analysis II**
## *Lab 3: Signal vs Background*

Last laboratory we learned how to employ variable cuts on our dataset in order to better populate our signal region. Today we will also introduce background processes, which will pollute our data. In order to minimize this background, we will need to look at the observables stored in our datasets and choose the best points to perform the cuts.

In [ ]:
import atlasopenmagic as atom
import awkward as ak
import uproot
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt
import vector
import os
import requests

Luck has it that our $Z → μ^+μ^-$ process is extremely clean and easy to differentiate, with few background processes that mimic the final states. The more substantial ones that might appear are the following:

 - $t$ quark production: both pair production and single $t$ can yield similar final states
 - Drell-Yan continuum: any pairs of leptons that are produced via $Z/γ^*$ decays
 - $V\ V$ production: combinations of vector boson pair production can yield multi-lepton final states
 - Higgs production with vector boson decays
 - $W$+jets processes

The dataset IDs for the processes are provided here. We will now also work with actual data, so we added it to our dictionary.

A smarter approach that we will use from now on is accessing the data via the `root://` protocol, which natively works with `.root` files and does not cache locally the same way `https://` does. This will prove very helpful when working with actualy data since the sets are very large.

In [ ]:
atom.set_release('2025e-13tev-beta')

defs = {
    r'Data': {'dids': ['data']},
    r'Z2mumu': {'dids': [700323, 700324, 700325, 700359]},
    r'top': {'dids': [410470, 410471, 411234, 410155, 410156, 410157]},
    r'Higgs': {'dids': [345324, 345066]},
    r'drell-yan': {'dids': [700320, 700322, 700360, 700792, 700793, 700794]},
    r'diboson': {'dids': [700488, 700489, 700490, 700492, 700493, 700494, 700589, 700590, 700592, 700593, 700602, 700603]},
    r'w_jets': {'dids': [700338, 700339, 700340, 700341, 700342, 700343]}
}

# We can force atlasopenmagic to use the root:// protocol, which natively opens root files without caching them
# We will use the 2 muon final state skim to get smaller datasets
data_samples = atom.build_dataset(defs, skim='2muons', protocol='root')

### Monte Carlo normalization

The data we have been working with so far was generated using Monte Carlo methods. As such, they are ideal and do not account for many effects that appear in reality. As such, we need to introduce the so-called "weights" that bring the simulations closer to the real data. We will compute these weights, taking into considerations factors such as:
 - Production cross-sections (`xsec`)
 - Pile-up corrections (`ScaleFactor_PILEUP`)
 - Trigger efficiencies (`ScaleFactor_MuTRIGGER`)
 - b-tagging efficiencies (`ScaleFactor_FTAG`)
 - Monte Carlo weights (`mcWeight`)

In [ ]:
def calc_weight(data):
    weight_list =( data["ScaleFactor_MUON"] * data["ScaleFactor_MuTRIGGER"] * data["ScaleFactor_PILEUP"] * data["ScaleFactor_FTAG"] *
             ( data["mcWeight"] / data["sum_of_weights"]) * (data["xsec"] * data["filteff"] * data["kfac"] * lumi) )
    return weight_list

Here we will define the cuts we want to perform on our data, similar to what we did in the last laboratory! I wrote some suggestions here, you will need to implement them, tweak them and if you want, add more.

In [ ]:
# Check if the muon trigger has fired in the event
def cut_trigM(trigM):
    return 

# Check if at least one lepton in the event is matched with a trigger
def cut_isTrigMatched(isTrigMatched):
    return 

# Cut on the number of leptons in an event
def cut_lep_n(lep_n):
    return

# Cut on the charge of the leptons in an event
def cut_lep_charge(lep_charge):
    return

# Cut on the type of the leptons in an event (look at their IDs in the PDG)
def cut_lep_type(lep_type):
    return 

# Cut on the transverse momenta of the leptons
def cut_lep_pt(lep_pt):
    return 

# Cut on the correct identification of the particle and on whether the detected muons are well isolated
def cut_ID_Iso(lep_isMediumID, lep_isLooseIso):
    return

# Cut on the missing transverse energy of the event
def cut_met(met):
    return 

# Cut on the number of jets in the event
def cut_jet_n(jet_n):
    return 

# And other cuts that you see fit and want to implement!

We also need to compute the invariant mass of our leptons in order to see if we successfully reconstructed the $Z$ boson.

In [ ]:
def calc_mass(lep_pt, lep_eta, lep_phi, lep_e): 
    p4 = vector.zip({
        'pt': lep_pt,
        'eta': lep_eta,
        'phi': lep_phi,
        'E': lep_e
    })
    return # Invariant mass

Since we have multiple datasets, it is easier to define a function that parses each file, performs cuts and keeps the relevant data.

Here we will perform the cuts, compute the invariant mass, compute the weights and store everything.

In [ ]:
def process_file(filename, sample_name):
    tree = uproot.open(filename + ':analysis')
    sample_data = []

    for data in tree.iterate(filter_name=variables, library='ak', entry_stop=int(fraction*tree.num_entries), step_size=100_000):
        # data = Perform the cuts on the dataset here

        # data["inv_mass"] = Compute invariant mass and store it

        # data["Weight"] = Compute weights and store them
        # You will need to be careful to weigh only the MC samples and not the real data!
        
        sample_data.append(data)

    if sample_data is not None and len(sample_data) > 0:
        return ak.concatenate(sample_data, axis=0) 
    else:
        return ak.Array([])

We will choose the variables, define the luminosity and start processing the data files. We define the `fraction` factor to only go through a certain section of the datasets, as to not wait that long.

In [ ]:
variables = [
    "mcWeight", "ScaleFactor_MuTRIGGER", "trigM", "lep_n", "lep_pt", "lep_eta", "lep_phi", 
    "lep_e", "lep_charge", "lep_type", "met", "met_phi", "lep_isMediumID", "lep_isLooseIso", 
    "sum_of_weights", "ScaleFactor_FTAG", "xsec", "jet_pt", "jet_n", "jet_eta", "jet_phi", 
    "jet_e", "lep_isTrigMatched", "eventNumber", "ScaleFactor_ELE", "ScaleFactor_MUON", 
    "ScaleFactor_PILEUP", "filteff", "kfac"
]

# Dictionary storing the data per process
data_all = {}
# Luminosity in pb^-1 You can find the luminosity of each dataset on the ATLAS Open Data Platform
lumi = 36600
# Fraction of the dataset to process
fraction = 0.1

for sample in data_samples:
    frames = []
    print(f"Processing the {sample} sample")

    for file in data_samples[sample]["list"]:
        df = process_file(file, sample)
        if df is not None and len(df) > 0:
            frames.append(df)

    data_all[sample] = ak.concatenate(frames, axis=0)

You can use this cell to visualize the various variables per process, in order to choose the point were the cut on the variable is performed.

In [ ]:
# For jagged arrays, we need to broadcast the weights to match the entries 
# E.g. pT, eta, phi, charge etc., variables where there can be multiple entries per event
datasets = [
    {'data': ak.flatten(data_all["w_jets"]['lep_pt']), 'weights': ak.flatten(ak.broadcast_arrays(data_all["w_jets"]["lep_pt"], data_all["w_jets"]['Weight'])[1]), 'color': 'thistle', 'label': r'$W$+jets'},
    {'data': ak.flatten(data_all["Higgs"]['lep_pt']), 'weights': ak.flatten(ak.broadcast_arrays(data_all["Higgs"]["lep_pt"], data_all["Higgs"]['Weight'])[1]), 'color': 'lightsalmon', 'label': r'Higgs'},
    {'data': ak.flatten(data_all["top"]['lep_pt']), 'weights': ak.flatten(ak.broadcast_arrays(data_all["top"]["lep_pt"], data_all["top"]['Weight'])[1]), 'color': 'lightcyan', 'label': r'Top production'},
    {'data': ak.flatten(data_all["drell-yan"]["lep_pt"]), 'weights': ak.flatten(ak.broadcast_arrays(data_all["drell-yan"]["lep_pt"], data_all["drell-yan"]['Weight'])[1]), 'color': 'palegreen', 'label': r'Drell-Yan'},
    {'data': ak.flatten(data_all["diboson"]['lep_pt']), 'weights': ak.flatten(ak.broadcast_arrays(data_all["diboson"]["lep_pt"], data_all["diboson"]['Weight'])[1]), 'color': 'moccasin', 'label': r'Diboson'},
    {'data': ak.flatten(data_all["Z2mumu"]['lep_pt']), 'weights': ak.flatten(ak.broadcast_arrays(data_all["Z2mumu"]["lep_pt"], data_all["Z2mumu"]['Weight'])[1]), 'color': 'cornflowerblue', 'label': r'$Z→μμ$'},
]

# For flat arrays we don't need to broadcast the weights
# E.g. lepton/jet numbers, MET, variables that are global per event
# datasets = [
#     {'data': data_all["w_jets"]['lep_n'], 'weights': data_all["w_jets"]['Weight'], 'color': 'thistle', 'label': r'$W$+jets'},
#     {'data': data_all["Higgs"]['lep_n'], 'weights': data_all["Higgs"]['Weight'], 'color': 'lightsalmon', 'label': r'Higgs'},
#     {'data': data_all["top"]['lep_n'], 'weights': data_all["top"]['Weight'], 'color': 'lightcyan', 'label': r'Top'},
#     {'data': data_all["drell-yan"]["lep_n"], 'weights': data_all["drell-yan"]['Weight'], 'color': 'palegreen', 'label': r'Drell-Yan'},
#     {'data': data_all["diboson"]['lep_n'], 'weights': data_all["diboson"]['Weight'], 'color': 'moccasin', 'label': r'Diboson'},
#     {'data': data_all["Z2mumu"]['lep_n'], 'weights': data_all["Z2mumu"]['Weight'], 'color': 'cornflowerblue', 'label': r'$Z→μμ$'},
# ]    

fig, ax = plt.subplots()
# The bin edges need to be adapted for each visualized variable
bins_edges = np.linspace(20, 150, 51)
bins_centers = (bins_edges[1:] + bins_edges[:-1]) / 2
ax.hist([d['data'] for d in datasets], bins=bins_edges, weights=[d['weights'] for d in datasets], stacked=True, color=[d['color'] for d in datasets], label=[d['label'] for d in datasets])
ax.set_ylabel('Events')
ax.legend()
ax.set_yscale("log")
plt.show()

Now we can take a look at the invariant mass spectrum of the 2 muons and see if we successfully reconstructed the $Z$ mass.

In [ ]:
datasets = [
    {'data': data_all["w_jets"]['inv_mass'], 'weights': data_all["w_jets"]['Weight'], 'color': 'thistle', 'label': r'$W$+jets'},
    {'data': data_all["Higgs"]['inv_mass'], 'weights': data_all["Higgs"]['Weight'], 'color': 'lightsalmon', 'label': r'Higgs'},
    {'data': data_all["top"]['inv_mass'], 'weights': data_all["top"]['Weight'], 'color': 'lightcyan', 'label': r'Top'},
    {'data': data_all["drell-yan"]["inv_mass"], 'weights': data_all["drell-yan"]['Weight'], 'color': 'palegreen', 'label': r'Drell-Yan'},
    {'data': data_all["diboson"]['inv_mass'], 'weights': data_all["diboson"]['Weight'], 'color': 'moccasin', 'label': r'Diboson'},
    {'data': data_all["Z2mumu"]['inv_mass'], 'weights': data_all["Z2mumu"]['Weight'], 'color': 'cornflowerblue', 'label': r'$Z→μμ$'},
]

fig, ax = plt.subplots()

bins_mass = np.arange(10, 250, 5)
bins_mass_centers = (bins_mass[1:] + bins_mass[:-1]) / 2

# This is for the real data points
data_x, _ = np.histogram(ak.to_numpy(data_all['Data']['inv_mass']), bins=bins_mass)
data_x_err = np.sqrt(data_x)
ax.errorbar(bins_mass_centers, data_x, yerr=data_x_err, fmt='ko', label='Data')

# This is for the MC samples
ax.hist([d['data'] for d in datasets], bins=bins_mass, weights=[d['weights'] for d in datasets], stacked=True, color=[d['color'] for d in datasets], label=[d['label'] for d in datasets])

ax.set_xlabel(r'$m_{\mu\mu}$ [GeV]')
ax.set_ylabel(f'Events / {(bins_mass[1]-bins_mass[0]):.1f} GeV')
ax.legend()
ax.set_yscale("log")
plt.show()

### Improvements that can be made

Now that we have some results, we can:
 - Fit the distribution and see what the mean $M_Z$ is
 - Further fine-tune the cuts
 - Extract the number of signal and bacgkround events and see what $S/\sqrt{B}$ we have in the signal region.

Choose what how you want to proceed and try to implement it!